In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
import torch
import torch.nn.functional as F


class DiffusionQuotaHelper(ABC):
    @abstractmethod
    def get_quota(self, step_current):
        pass
    # end
# end

class BlockDiffusionQuotaHelper(DiffusionQuotaHelper):
    def __init__(self, block_mask_index: torch.Tensor, steps_per_block: int) -> torch.Tensor:
        device = block_mask_index.device
        dtype = torch.long

        total = block_mask_index.sum(dim=1)                  # (B,)
        base  = torch.div(total, steps_per_block, rounding_mode='floor')  # (B,)
        rem   = total - base * steps_per_block                         # (B,)

        # Start with base for all steps
        num_transfer_tokens = base.unsqueeze(1).expand(-1, steps_per_block).to(dtype)  # (B, steps)

        # Add +1 to the first `rem[b]` steps for each batch b — without tensor slicing
        cols = torch.arange(steps_per_block, device=device).unsqueeze(0)               # (1, steps)
        add_mask = cols < rem.unsqueeze(1)                                   # (B, steps)
        self.num_transfer_tokens = num_transfer_tokens + add_mask.to(dtype)       # (B, steps)
    # end

    def get_quota(self, step_current):
        quota_current = self.num_transfer_tokens[:, step_current]

        if quota_current.dim() == 2 and quota_current.size(1) == 1:
            quota_current = quota_current.squeeze(1)
        # end

        return quota_current
    # end
# end



class ConfKSorter:

    def argsort(self, confidence):
        idx_sorted = torch.argsort(confidence, dim=1, descending=True)
        return idx_sorted
    # end
# end

class RandomKSorter(ConfKSorter):
    def argsort(self, confidence, snapshot):

        confidence = torch.where(
                snapshot.mask_mask,
                torch.rand(confidence.shape[0], confidence.shape[1], device=confidence.device),
                confidence
            )

        return self.super(confidence)

    # end
# end


class TopKSorter(ConfKSorter):
    def argsort(self, confidence, snapshot):
        return self.super(confidence)
    # end
# end



class ConfCollector:
    def gather_x0_p(self, p, index):
        return torch.gather(p, dim=-1, index=index)
    # end
# end


class TruthCollector(ConfCollector):
    def gather_x0_p(self, p, snapshot):
        index = snapshot.y.unsqueeze(-1)
        return self.super(p, index)
    # end
# end


class MaxCollector(ConfCollector):
    def gather_x0_p(self, p, snapshot):
        index = snapshot.x0
        return self.super(p, index)
    # end
# end






class SimpleLogitsSnapshot:

    def __init__(self, logits, x, y, id_mask):
        self.logits = logits
        self.x = x
        self.y = y
        self.id_mask = id_mask
        self.mask_mask = (self.x == self.id_mask)
        self.x0 = torch.argmax(self.logits, dim=-1)
    # end


    def transform_logits(self, collector):
        p = F.softmax(self.logits.to(torch.float64), dim=-1)

        x0_p = collector.gather_x0_p(p, self)
        neg_inf = torch.tensor(torch.finfo(p.dtype).min, device=p.device, dtype=p.dtype)    # TODO: might be error here
        conf = torch.where(self.mask_mask, x0_p, neg_inf)

        return conf
    # end

    def argsort_confidence(self, conf, sorter): # TODO: 1. implement sorter; 2. think delayed / windowed refresh
        idx_sorted = sorter.sort(conf, self)
        return idx_sorted
    # end

    def generate_transfer_mask_by_quota(self, idx_sorted, quota):
        mask_mask = self.mask_mask

        B, L = idx_sorted.shape
        # Build a mask that is True for the first k[b] columns in each row (sorted order)
        cols = torch.arange(L, device=idx_sorted.device).unsqueeze(0).expand(B, L)   # (B, L)
        k_expanded = quota.unsqueeze(1).expand(B, L)                   # (B, L)
        select_sorted = cols < k_expanded                                            # (B, L) bool for top k
        transfer_int = torch.zeros(B, L, device=idx_sorted.device, dtype=torch.int8) # (B, L)
        transfer_mask = transfer_int.scatter(1, idx_sorted, select_sorted.to(torch.int8)).bool()

        mask_transfer = transfer_mask & mask_mask  # ensure we never select unmasked        

        return mask_transfer
    # end

    def _transfer_index_to_mask_by_quota(self, idx_sorted, quota):
        B, L = idx_sorted.shape
        # Build a mask that is True for the first k[b] columns in each row (sorted order)
        cols = torch.arange(L, device=idx_sorted.device).unsqueeze(0).expand(B, L)   # (B, L)
        k_expanded = quota.unsqueeze(1).expand(B, L)                   # (B, L)
        select_sorted = cols < k_expanded                                            # (B, L) bool for top k
        transfer_int = torch.zeros(B, L, device=idx_sorted.device, dtype=torch.int8) # (B, L)
        transfer_mask = transfer_int.scatter(1, idx_sorted, select_sorted.to(torch.int8)).bool()
    # end
# end



@dataclass
class DiffusionConfig:
    num_blocks: int
    step_per_block: int
    size_block: int
    id_mask: int
    len_prompt: int
    klass_sorter: ConfKSorter
    klass_collector: ConfCollector
# end


model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_with_snapshot(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector

    idx_refresh = torch.tensor([[]])

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)


        idx_denoising = x[:, position_start:position_end]
        x_current = torch.cat([idx_refresh, idx_denoising], dim=-1) 
        logits = model(x_current, idx_refresh=idx_refresh, idx_denoising=idx_denoising)

        snapshot = SimpleLogitsSnapshot(logits, x, y, id_mask)


        for step in range(step_per_block):
            conf = snapshot.transform_logits(collector)
            idx_sorted_by_conf = sorter.argsort(conf, snapshot)

            num_unmask = quota_helper.get_quota(step)
            idx_unmask = idx_sorted_by_conf[:, :num_unmask]

            if step == 0:
                snapshot.unmask_by_idx(idx_unmask)  # get the x0, x0_p
                idx_refresh = idx_unmask
                continue
            # end

            idx_denoising = idx_unmask
            idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1)   # 原本的位置
            x_current = x[:, idx_current]
            logits = model(x_current, idx_current=idx_current) # 有问题兄弟

            logits_refresh = logits[:,:idx_refresh.shape[-1]]
            logits_denoising = logits[:, idx_denoising.shape[-1]:]

            snapshot.update_logits(logits_refresh, idx_refresh)
            snapshot.unmask_by_logits(logits_denoising, idx_denoising)

            idx_refresh = idx_denoising
        # end
    # end
# end